# Phonecode Demo

This notebook demonstrates how to create and use phoneme-based coordinate encodings for Codenames.

## Setup

First, download the required data (CMUdict). This is a one-time setup:

In [ ]:
from tapyoca import phonecode

# One-time setup - download required data
phonecode.download_required_data()

## Create the Mapping

Create a 5×5 grid mapping with words for:
- Single coordinates (25 total)
- 2-coordinate combinations (up to 300 possible)

In [15]:
# Create mapping with both single and 2-coord words
result = phonecode.create_5x5_codenames_mapping(
    words_per_coord=2,      # Find 2 words per single coordinate
    check_multi_coord=True, # Enable 2-coord combinations
    max_multi_coord=2       # Limit to pairs
)

Finding words for single coordinates...
Coverage: 25/25 coordinates have at least one word

Searching for words encoding multiple coordinates...
  Checking 2-coordinate combinations...
Coverage: 25/25 coordinates have at least one word

Searching for words encoding multiple coordinates...
  Checking 2-coordinate combinations...
    Found words for 300 combinations
    Found words for 300 combinations


## Display Single-Coordinate Words

Show words for each coordinate in a 5×5 grid:

In [16]:
# Display the 25 single-coordinate words in a grid
phonecode.print_coord_words_grid(result, max_words_per_cell=3)


WORD GRID (Consonant-Vowel Combinations)

Consonants (rows): ['P', 'T', 'K', 'F', 'S']
Vowels (columns): ['IY', 'AE', 'AA', 'OW', 'UW']

                IY                AE                AA                OW                UW        
        ------------------------------------------------------------------------------------------
     P |      people           program             upon            program            group       
             company           example             part           production         produced     
        ··························································································
     T |     between             that              not               most              into       
             present            after              part             total              two        
        ··························································································
     K |     company             can             because           con

## Display 2-Coordinate Word Summary

Show statistics about 2-coordinate combinations:

In [17]:
# Show coverage summary for 2-coord combinations
phonecode.print_two_coord_summary(result)


2-COORDINATE WORD COVERAGE SUMMARY

Total possible 2-coord combinations: 300
Combinations with at least 1 word:   300 (100.0%)
Total words found:                    1432
Average words per combination:        4.8

--- Top 10 Best Covered Combinations ---
  ((0, 0), (0, 1))     →   5 words
  ((0, 0), (0, 2))     →   5 words
  ((0, 0), (0, 3))     →   5 words
  ((0, 0), (0, 4))     →   5 words
  ((0, 0), (1, 0))     →   5 words
  ((0, 0), (1, 1))     →   5 words
  ((0, 0), (1, 2))     →   5 words
  ((0, 0), (1, 3))     →   5 words
  ((0, 0), (1, 4))     →   5 words
  ((0, 0), (2, 0))     →   5 words

--- Bottom 10 (Least Covered) ---
  ((2, 3), (3, 4))     →   2 words
  ((2, 4), (3, 3))     →   2 words
  ((0, 1), (3, 4))     →   1 words
  ((0, 2), (3, 4))     →   1 words
  ((0, 3), (3, 4))     →   1 words
  ((0, 4), (3, 1))     →   1 words
  ((0, 4), (3, 2))     →   1 words
  ((0, 4), (3, 3))     →   1 words
  ((3, 3), (4, 4))     →   1 words
  ((3, 4), (4, 3))     →   1 words


## Display 2-Coordinate Words Table

Show detailed list of words that encode 2 coordinates:

In [ ]:
# Display detailed table of 2-coord words
phonecode.print_two_coord_words_table(result, max_words_per_combo=5, max_combos=20)

## Verify Word Decoding

Use the codec to decode words back to coordinates and verify they're correct:

In [23]:
# Create a codec for encoding/decoding
codec = phonecode.CoordsWordCodec(result['mapping'], result['single_coord_words'])

# Test decoding a single-coordinate word
word = "peace"  # Should decode to coord with P and IY
decoded_coords, is_valid = codec.decode_word_to_coords(word)

print(f"Word: '{word}'")
print(f"Decoded to coords: {decoded_coords}")
print(f"Phoneme pairs: {[result['mapping'].coord_to_phonemes[c] for c in decoded_coords]}")

Word: 'peace'
Decoded to coords: [(0, 0)]
Phoneme pairs: [('P', 'IY')]


In [27]:
# Now let's get a REAL 2-coordinate word from our results
if 'multi_coord_words' in result and 2 in result['multi_coord_words']:
    two_coord_dict = result['multi_coord_words'][2]
    if two_coord_dict:
        # Get first combination that has words
        first_combo = next(iter(two_coord_dict.items()))
        phoneme_pairs, words = first_combo
        
        if words:
            test_word = words[0][0]  # First word
            test_phonemes = words[0][1]  # Its phonemes
            
            # Decode it
            decoded_coords, is_valid = codec.decode_word_to_coords(test_word)
            
            print(f"Real 2-coordinate word: '{test_word}'")
            print(f"Phonemes: {test_phonemes}")
            print(f"Decoded to coords: {decoded_coords}")
            for coord in decoded_coords:
                pair = result['mapping'].coord_to_phonemes[coord]
                print(f"  Coord {coord} → {pair}")
            print(f"\n✓ Successfully decoded {len(decoded_coords)} coordinates!")
else:
    print("No 2-coordinate words found in result")

Real 2-coordinate word: 'capacity'
Phonemes: ['K', 'AH', 'P', 'AE', 'S', 'AH', 'T', 'IY']
Decoded to coords: [(2, 1), (4, 0)]
  Coord (2, 1) → ('K', 'AE')
  Coord (4, 0) → ('S', 'IY')

✓ Successfully decoded 2 coordinates!


## Validate Specific Encodings

Verify that a word correctly encodes expected coordinates:

## Find Words for Specific Coordinates

Search for words that encode a specific pair of coordinates:

In [25]:
# Find words that encode specific coordinates
target_coords = [(0, 0), (1, 1)]  # P-IY and T-AE

print(f"Finding words for coords {target_coords}:")
print(f"Phoneme pairs needed: {codec.encode_coords_to_phoneme_pairs(target_coords)}")

# Find valid words
valid_words = codec.find_valid_words_for_coords(target_coords, max_results=5)

print(f"\nFound {len(valid_words)} words:")
for word, phonemes, freq in valid_words:
    # Verify decoding
    decoded, is_valid = codec.decode_word_to_coords(word, expected_num_coords=2)
    status = "✓" if is_valid and set(decoded) == set(target_coords) else "✗"
    print(f"  {status} {word:15s} → {decoded}")

Finding words for coords [(0, 0), (1, 1)]:
Phoneme pairs needed: [('P', 'IY'), ('T', 'AE')]

Found 0 words:
